# FOMC statements -- postprocess (the clean, tidy corpus)

Takes the raw compiled dataset and produces the final analysis-ready
dataset: one row per document, cleaned text, plus metadata to tell apart
the main post-meeting policy statement from companion documents that share
the same date (e.g. a "Statement on Longer-Run Goals" reaffirmation).

Cleaning fixes applied here (see Helpers/fomc_helpers.py clean_fomc_statement
and PROGRESS_LOG.md for what each one fixes and why):
- "Share" button text and the "For release at [time]" line stripped.
- Old-era (1994-2007ish) page-navigation footer stripped.
- Two footer-stripping rules that were silently deleting real policy content
  ("in a related action...", "reserve banks...discount rate...") removed --
  they matched substantive mid-document text, not true boilerplate.

In [1]:
import os, sys
import pandas as pd
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "Helpers"))
from fomc_helpers import clean_fomc_statement, url_doc_letter

BASE = os.path.dirname(os.getcwd())
COMPILED_PATH = os.path.join(BASE, "Data Input", "Raw Input", "fomc_statements_compiled_raw.csv")
CLEAN_PATH = os.path.join(BASE, "Data Output", "fomc_statements_clean.csv")

## Load the compiled raw dataset

In [2]:
df = pd.read_csv(COMPILED_PATH, parse_dates=["date"])
print(df.shape)

(264, 9)


## Clean the text

In [3]:
df["clean_text"] = df["raw_text"].map(lambda s: clean_fomc_statement(s, min_chars=30, flatten=False))
df["clean_word_count"] = df["clean_text"].str.split().str.len()

## Derive document-subtype metadata

The URL's trailing letter before the extension ("a"/"b"/"c"...) reliably
marks the main post-meeting statement ("a") vs. companion documents
released the same day. page_title gives each document its own real,
specific label (e.g. "Statement Regarding Monetary Policy Implementation
and Balance Sheet Normalization") instead of a generic "Statement" bucket.

In [4]:
df["doc_letter"] = df["url"].map(url_doc_letter)
df["is_main_statement"] = df["doc_letter"] == "a"
df["title"] = df["page_title"].str.replace(r"^Federal Reserve Board\s*-\s*", "", regex=True)
print(df["is_main_statement"].value_counts())

is_main_statement
True     249
False     15
Name: count, dtype: int64


## Final tidy columns and save

In [5]:
speaker_fomc = None
tidy = pd.DataFrame({
    "date": df["date"],
    "year": df["year"],
    "speaker": speaker_fomc,
    "doc_type": "FOMC_statement",
    "title": df["title"],
    "is_main_statement": df["is_main_statement"],
    "source_corpus": "fomc_statements",
    "url": df["url"],
    "word_count": df["clean_word_count"],
    "clean_text": df["clean_text"],
    "raw_text": df["raw_text"],
    "doc_letter": df["doc_letter"],
})
tidy = tidy.sort_values(["date", "doc_letter"]).drop(columns="doc_letter").reset_index(drop=True)
os.makedirs(os.path.dirname(CLEAN_PATH), exist_ok=True)
tidy.to_csv(CLEAN_PATH, index=False)
print(f"Saved {len(tidy):,} rows to: {CLEAN_PATH}")
print(f"word_count: min={tidy['word_count'].min()}, max={tidy['word_count'].max()}, median={tidy['word_count'].median()}")

Saved 264 rows to: C:\Users\User\Desktop\DISSERTATION PAPERS\FED_TREASURY_DATA\Data Output\fomc_statements_clean.csv
word_count: min=39, max=806, median=285.0
